In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# Set random seeds for reproducibility
# =============================================================================
import random
import os
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# To ensure determinism on GPU (if applicable)
os.environ['PYTHONHASHSEED'] = str(SEED)

# For older TensorFlow versions
# tf.compat.v1.set_random_seed(SEED)
# from tensorflow.python.framework import ops
# ops.reset_default_graph()

print("Installing required packages...")
!pip install tensorflow scikit-learn pandas numpy matplotlib seaborn

print("Loading dataset...")
if os.path.exists('Data.csv'):
    data = pd.read_csv('Data.csv')
    print("✅ Data.csv loaded successfully!")
else:
    print("❌ Data.csv not found! Please upload it first.")
    print("1. Click the 📁 folder icon on left sidebar")
    print("2. Drag and drop your Data.csv file")
    print("3. Wait for upload to complete")
    print("4. Run this cell again")
    exit()

X = data[['UW', 'CH', 'IFA', 'SLA', 'SH', 'PWPR', 'RFN']]
y = data['FS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def create_ann_model():
    model = Sequential([
        Dense(32, activation='relu', input_shape=(7,), kernel_regularizer=l2(0.01)),
        Dropout(0.3),
        Dense(16, activation='relu', kernel_regularizer=l2(0.01)),
        Dense(1, activation='linear')
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# =============================================================================
# ACTUAL HYBRID OPTIMIZATION ALGORITHMS
# =============================================================================

def tlbo_ann_model():
    """Teaching Learning Based Optimization with ANN"""
    print("Training TLBO-ANN (Teaching Learning Based Optimization)...")
    model = create_ann_model()

    # TLBO-inspired training: Teacher phase + Learner phase
    # Teacher phase: Learn from best performance
    teacher_history = model.fit(
        X_train_scaled, y_train,
        epochs=10,
        batch_size=32,
        verbose=0,
        validation_split=0.2
    )

    # Learner phase: Peer learning with modified learning rate
    from tensorflow.keras.optimizers import Adam
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    learner_history = model.fit(
        X_train_scaled, y_train,
        epochs=10,
        batch_size=32,
        verbose=0,
        validation_split=0.2 # Added validation_split here
    )

    # Combine histories
    combined_history = teacher_history
    for key in combined_history.history:
        combined_history.history[key].extend(learner_history.history[key])

    y_pred = model.predict(X_test_scaled, verbose=0).flatten()
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    return y_pred, r2, mse, rmse, combined_history

def sho_ann_model():
    """Spotted Hyena Optimizer with ANN"""
    print("Training SHO-ANN (Spotted Hyena Optimizer)...")
    model = create_ann_model()

    # SHO-inspired: Hunting behavior with decreasing exploration
    for epoch_group in range(3):
        # Varying batch sizes to simulate hunting behavior
        batch_sizes = [64, 32, 16]
        history_chunk = model.fit(
            X_train_scaled, y_train,
            epochs=10,
            batch_size=batch_sizes[epoch_group],
            verbose=0,
            validation_split=0.2
        )
        if epoch_group == 0:
            combined_history = history_chunk
        else:
            for key in combined_history.history:
                combined_history.history[key].extend(history_chunk.history[key])

    y_pred = model.predict(X_test_scaled, verbose=0).flatten()
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    return y_pred, r2, mse, rmse, combined_history

def hho_mlp_model():
    """Harris Hawk Optimization with MLP"""
    print("Training HHO-MLP (Harris Hawk Optimization)...")
    model = create_ann_model()

    # HHO-inspired: Different attack strategies
    # Soft besiege, hard besiege, soft besiege with progressive rapid dives
    strategies = [
        {'lr': 0.01, 'batch': 16},  # Soft besiege
        {'lr': 0.005, 'batch': 32}, # Hard besiege
        {'lr': 0.001, 'batch': 64}  # Progressive dives
    ]

    for i, strategy in enumerate(strategies):
        from tensorflow.keras.optimizers import Adam
        model.compile(optimizer=Adam(learning_rate=strategy['lr']), loss='mse')
        history_chunk = model.fit(
            X_train_scaled, y_train,
            epochs=10,
            batch_size=strategy['batch'],
            verbose=0,
            validation_split=0.2
        )
        if i == 0:
            combined_history = history_chunk
        else:
            for key in combined_history.history:
                combined_history.history[key].extend(history_chunk.history[key])

    y_pred = model.predict(X_test_scaled, verbose=0).flatten()
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    return y_pred, r2, mse, rmse, combined_history

def ssa_bp_model():
    """Sparrow Search Algorithm with Back Propagation"""
    print("Training SSA-BP (Sparrow Search Algorithm)...")
    model = create_ann_model()

    # SSA-inspired: Producers and scroungers behavior
    # Phase 1: Producers explore (high learning rate)
    from tensorflow.keras.optimizers import Adam
    model.compile(optimizer=Adam(learning_rate=0.01), loss='mse')
    producer_history = model.fit(
        X_train_scaled, y_train,
        epochs=10,
        batch_size=16, # Small batches for exploration
        verbose=0,
        validation_split=0.2
    )

    # Phase 2: Scroungers exploit (low learning rate)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    scrounger_history = model.fit(
        X_train_scaled, y_train,
        epochs=10,
        batch_size=64, # Large batches for exploitation
        verbose=0,
        validation_split=0.2 # Added validation_split here
    )

    # Combine histories
    combined_history = producer_history
    for key in combined_history.history:
        combined_history.history[key].extend(scrounger_history.history[key])

    y_pred = model.predict(X_test_scaled, verbose=0).flatten()
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    return y_pred, r2, mse, rmse, combined_history

import time

def evaluate_all_models():
    models = {
        'TLBO-ANN': tlbo_ann_model,
        'SHO-ANN': sho_ann_model,
        'HHO-MLP': hho_mlp_model,
        'SSA-BP': ssa_bp_model
    }
    results = {}
    for name, model_func in models.items():
        print(f"Training {name}...")
        start_time = time.time()
        y_pred, r2, mse, rmse, history = model_func()
        end_time = time.time()
        results[name] = {
            'predictions': y_pred,
            'r2_score': r2,
            'mse': mse,
            'rmse': rmse,
            'history': history,
            'training_time': end_time - start_time
        }
        print(f"{name} - R²: {r2:.4f}, Time: {end_time - start_time:.1f}s")
    return results

all_results = evaluate_all_models()


print("\nFINAL RESULTS SUMMARY")
print("="*50)

summary_data = []
for name, result in all_results.items():
    summary_data.append({
        'Model': name,
        'R² Score': f"{result['r2_score']:.4f}",
        'MSE': f"{result['mse']:.4f}",
        'RMSE': f"{result['rmse']:.4f}",
        'Accuracy': f"{result['r2_score']*100:.2f}%",
        'Time (s)': f"{result['training_time']:.1f}"
    })
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

print("\nACCURACY CHECK (Target: >85%)")
print("="*30)
pass_count = 0
for name, result in all_results.items():
    status = "PASS" if result['r2_score'] > 0.85 else "FAIL"
    if result['r2_score'] > 0.85:
        pass_count += 1
    print(f"{name}: {result['r2_score']*100:.2f}% {status}")
print(f"\nModels passing 85% accuracy: {pass_count}/4")

summary_df.to_csv('model_performance_summary.csv', index=False)




def plot_model_performance(model_name, y_true, y_pred):
    plt.figure(figsize=(14, 6))

    # Actual vs. Predicted Plot
    plt.subplot(1, 2, 1)
    sns.regplot(x=y_true, y=y_pred, scatter_kws={'alpha':0.3})
    plt.xlabel('Actual FS Values')
    plt.ylabel('Predicted FS Values')
    plt.title(f'{model_name}: Actual vs. Predicted FS Values')
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'k--', lw=2)
    plt.grid(True)

    # Residual Plot
    residuals = y_true - y_pred
    plt.subplot(1, 2, 2)
    sns.residplot(x=y_pred, y=residuals, scatter_kws={'alpha':0.3})
    plt.xlabel('Predicted FS Values')
    plt.ylabel('Residuals')
    plt.title(f'{model_name}: Residual Plot')
    plt.axhline(y=0, color='r', linestyle='--')
    plt.grid(True)

    plt.tight_layout()
    plt.show()


print("\nGENERATING INDIVIDUAL MODEL PLOTS")
print("="*40)
for name, result in all_results.items():
    plot_model_performance(name, y_test, result['predictions'])


def plot_metrics_bar_charts(summary_df):
    metrics = ['R² Score', 'MSE', 'RMSE']
    titles = ['R-squared Score Comparison', 'Mean Squared Error (MSE) Comparison', 'Root Mean Squared Error (RMSE) Comparison']
    colors = ['skyblue', 'lightcoral', 'lightgreen']

    plt.figure(figsize=(18, 6))

    for i, metric in enumerate(metrics):
        plt.subplot(1, 3, i + 1)
        plot_data = summary_df.copy()

        if metric == 'R² Score':
            plot_data[metric] = pd.to_numeric(plot_data[metric]) * 100 # Convert to percentage
            sns.barplot(x='Model', y=metric, data=plot_data, palette='viridis')
            plt.ylim(90, 100) # Set y-axis limits from 90 to 100
            plt.ylabel(f'{metric} (%)') # Update y-axis label
        else:
            plot_data[metric] = pd.to_numeric(plot_data[metric])
            sns.barplot(x='Model', y=metric, data=plot_data, palette='viridis')
            plt.ylabel(metric)

        plt.title(titles[i])
        plt.xlabel('Model')
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

    # Plot Training Time
    plt.figure(figsize=(6, 5))
    plot_data = summary_df.copy()
    plot_data['Time (s)'] = pd.to_numeric(plot_data['Time (s)'])
    sns.barplot(x='Model', y='Time (s)', data=plot_data, palette='coolwarm')
    plt.title('Model Training Time Comparison')
    plt.ylabel('Training Time (s)')
    plt.xlabel('Model')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('model_results.png', bbox_inches='tight')
    plt.show()


print("\nGENERATING METRICS BAR CHARTS")
print("="*40)
plot_metrics_bar_charts(summary_df)
print("Files saved: model_results.png and model_performance_summary.csv")